# LES turbulent flow past a sphere

Run this notebook to generate the simulation outputs:
- mid-plane velocity & pressure
- wake centerline profile
- Smagorinsky eddy viscosity map

## Boundary conditions
- **Inlet**: fixed $U_\infty$ + mild turbulence intensity
- **Outlet**: convective (Orlanski) + soft pressure (non-reflecting)
- **Outer walls**: free-slip
- **Sphere**: no-slip

In [ ]:
import os
print("cwd:", os.getcwd())

In [ ]:
import numpy as np
import meshio

mesh_npz = np.load("data/processed_mesh.npz")
if "boundary_tag" not in mesh_npz.files:
    print("Adding boundary tags to data/processed_mesh.npz ...")
    unique_faces = mesh_npz["unique_faces"]
    face_lookup = {tuple(f): i for i, f in enumerate(unique_faces)}
    m = meshio.read("../fluid_mesh_3d.msh")
    names = {1: "inlet", 2: "outlet", 3: "walls", 4: "object"}
    faces = {k: [] for k in names.values()}
    boundary_tag = np.full(len(unique_faces), 5, dtype=np.int32)
    for block, tags in zip(m.cells[:-1], m.cell_data["gmsh:physical"][:-1]):
        for tri, tag in zip(block.data, tags):
            fi = face_lookup[tuple(sorted(tri))]
            faces[names[int(tag)]].append(fi)
            boundary_tag[fi] = int(tag)
    np.savez(
        "data/processed_mesh.npz",
        **{k: mesh_npz[k] for k in mesh_npz.files if k != "boundary_tag"},
        boundary_tag=boundary_tag,
        inlet_faces=np.array(faces["inlet"], dtype=np.int32),
        outlet_faces=np.array(faces["outlet"], dtype=np.int32),
        wall_faces=np.array(faces["walls"], dtype=np.int32),
        object_faces=np.array(faces["object"], dtype=np.int32),
    )
    print({k: len(v) for k, v in faces.items()})
else:
    tag = mesh_npz["boundary_tag"]
    print("boundary tags OK:", {n: int(np.sum(tag == i)) for n, i in
          [("inlet",1),("outlet",2),("walls",3),("object",4)]})

In [ ]:
from pathlib import Path
from les_sphere_flow import SphereLESSolver, SimConfig
from IPython.display import Image, display, Markdown

out = Path('outputs')
out.mkdir(exist_ok=True)

quick = True  # set False for longer run / smoother GIF

if quick:
    cfg = SimConfig(dt=1e-4, t_end=0.05, print_every=50, frame_every=10, inlet_ti=0.015)
    max_steps = 200
else:
    cfg = SimConfig(dt=1e-4, t_end=0.2, print_every=100, frame_every=25, inlet_ti=0.015)
    max_steps = 800

solver = SphereLESSolver('data/processed_mesh.npz', cfg)
solver.run(max_steps=max_steps, capture_gif=True)

solver.save_fields(str(out / 'les_sphere_fields.npz'))
outputs = {
    'midplane': solver.plot_midplane(str(out / 'flow_past_sphere_midplane.png')),
    'streamlines': solver.plot_streamlines(str(out / 'flow_past_sphere_streamlines.png')),
    'wake': solver.plot_wake_profile(str(out / 'flow_past_sphere_wake.png')),
    'nut': solver.plot_nu_t(str(out / 'flow_past_sphere_nut.png')),
    'gif': solver.save_gif(str(out / 'flow_past_sphere.gif')),
}
outputs


In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown('### Animation (GIF)'))
display(Image(filename=outputs['gif']))

display(Markdown('### Streamlines'))
display(Image(filename=outputs['streamlines']))

display(Markdown('### Still frames'))
for key in ('midplane', 'wake', 'nut'):
    print(outputs[key])
    display(Image(filename=outputs[key]))
